# Working with Claude

In [ ]:
import os
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/",
)

In [83]:
def build_context(context: str) -> dict:
    system_prompt = {
    "role": "system",
    "content": """
        Your task is to answer questions from the course participants
        based on the provided context. Keep the answers precise, no-nonsense and no frills
        
        Use the context to find relevant information and provide accurate
        answers. If the answer is not found in the context,
        respond with "I don't know".

        Context:
        {context}
    """
    }
    system_prompt["content"] = system_prompt["content"].format(context=context)
    return system_prompt

In [74]:
HAIKU_INPUT_PRICE_PER_MILLION = 1 / 1_000_000
HAIKU_OUTPUT_PRICE_PER_MILLION = 5 / 1_000_000

def llm(user_prompt, system_prompt):
    response = client.chat.completions.create(
        model="claude-haiku-4-5",
        messages=[
            system_prompt,
            {"role": "user", "content": user_prompt},
        ]
    )

    cost = (
        response.usage.prompt_tokens * HAIKU_INPUT_PRICE_PER_MILLION +
        response.usage.completion_tokens * HAIKU_OUTPUT_PRICE_PER_MILLION
    )
    return {
        "response": response.choices[0].message.content,
        "cost": cost,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    }

# Preparing Dataset

In [41]:
import requests

courses_url = "https://datatalks.club/faq/json/courses.json"
faq_url_prefix = "https://datatalks.club/faq"

def get(url: str) -> dict:
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

In [ ]:
from pydantic import BaseModel

class CourseModel(BaseModel):
    course: str
    course_name: str
    path: str
    questions_count: int


In [28]:
courses = [CourseModel(**c) for c in get(courses_url)]

In [30]:
courses[0].path

'/json/data-engineering-zoomcamp.json'

In [54]:
def get_faq(course: CourseModel):
    faq_url = f"{faq_url_prefix}/{course.path}"
    return get(faq_url)


In [55]:
documents = [
    doc
    for course in courses
    for doc in get_faq(course)
]

In [56]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

# Indexing with MinSearch

In [57]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [59]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

# RAG

In [95]:
def rag(question: str):
    search_results = search(question)
    ctx = build_context(search_results)
    answer = llm(user_prompt=question, system_prompt=ctx)
    return answer

In [96]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

{'response': 'Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.', 'cost': 0.000982, 'input_tokens': 817, 'output_tokens': 33}


In [97]:
rag("How do I get a certificate?")

{'response': 'To get a certificate, you need to:\n\n1. **Finish the course with a "live" cohort** - Certificates are only awarded for live cohorts, not for self-paced mode.\n\n2. **Pass the Capstone project** - This is the mandatory requirement. Homework is not required (though recommended), but you must complete and pass the Capstone.\n\n3. **Peer-review 3 capstones** - After submitting your project, you need to peer-review 3 other capstones. You can only do this while the course is running; the peer-review window closes after the course ends.',
 'cost': 0.001699,
 'input_tokens': 984,
 'output_tokens': 143}